In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:

import pandas as pd
import numpy as np
import torch
from transformers import pipeline
import wandb
import transformers


In [3]:
device = 0 if torch.cuda.is_available() else -1

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA: True
Tesla T4


In [4]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [5]:
# Map@3 utility
def map_at_3(correct, predictions):
    for k, pred in enumerate(predictions[:3]):
        if pred == correct:
            return 1.0 / (k + 1)
    return 0.0

In [6]:
# Load Zero-Shot Classifier
classifier = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=device
)

def predict_top3(row):

    prompt = str(row.prompt)

    options = {
        "A": str(row.A),
        "B": str(row.B),
        "C": str(row.C),
        "D": str(row.D),
        "E": str(row.E)
    }

    result = classifier(
        prompt,
        candidate_labels=list(options.values()),
        hypothesis_template="The correct answer to the question is: {}"
    )

    label_to_opt = {v: k for k, v in options.items()}

    ranked = [label_to_opt[label] for label in result["labels"]]

    return " ".join(ranked[:3])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [7]:
# Validate on TRAIN first
print("Validating on training set...")
scores = []
for i, row in enumerate(train.itertuples(index=False)):
    pred = predict_top3(row)
    scores.append(map_at_3(row.answer, pred.split()))
    if i % 100 == 0:
        running = round(sum(scores) / len(scores), 4)
        print(f"  [{i}/{len(train)}] Running MAP@3: {running}")

train_map = round(sum(scores) / len(scores), 4)
print(f"\nFinal Train MAP@3: {train_map}")

Validating on training set...
  [0/2000] Running MAP@3: 0.3333


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [100/2000] Running MAP@3: 0.5693
  [200/2000] Running MAP@3: 0.6036
  [300/2000] Running MAP@3: 0.5919
  [400/2000] Running MAP@3: 0.5819
  [500/2000] Running MAP@3: 0.5915
  [600/2000] Running MAP@3: 0.5771
  [700/2000] Running MAP@3: 0.5718
  [800/2000] Running MAP@3: 0.5745
  [900/2000] Running MAP@3: 0.5744
  [1000/2000] Running MAP@3: 0.5728
  [1100/2000] Running MAP@3: 0.5658
  [1200/2000] Running MAP@3: 0.5669
  [1300/2000] Running MAP@3: 0.567
  [1400/2000] Running MAP@3: 0.5661
  [1500/2000] Running MAP@3: 0.5661
  [1600/2000] Running MAP@3: 0.564
  [1700/2000] Running MAP@3: 0.564
  [1800/2000] Running MAP@3: 0.5627
  [1900/2000] Running MAP@3: 0.5652

Final Train MAP@3: 0.5665


In [8]:
# Generating submission
print("Generating predictions...")
predictions = []
for i, row in enumerate(test.itertuples(index=False)):  
    pred = predict_top3(row)
    predictions.append({'id': row.id, 'prediction': pred})
    if i % 50 == 0:
        print(f"  {i}/{len(test)} done...")

submission = pd.DataFrame(predictions)
submission.to_csv('submission.csv', index=False)


Generating predictions...
  0/500 done...
  50/500 done...
  100/500 done...
  150/500 done...
  200/500 done...
  250/500 done...
  300/500 done...
  350/500 done...
  400/500 done...
  450/500 done...


In [ ]:
'''
This submission uses a pre-trained Zero-Shot Classification model to predict the correct answers.

For each question, the model takes the question and all five answer options as input. 
It checks which answer best matches the question and gives a score to each option. 
The options are then arranged from highest score to lowest score, and the top three options are used as the final prediction.

Unlike the previous methods, this approach does not depend only on matching words. 
Instead, it tries to understand the question and decide which answer is most suitable.

By using this method, I obtained a score of 0.49958.

The Conclusion: This method gave better predictions than the earlier baseline approaches because it uses a stronger pre-trained language model. 
However, some questions still require deeper reasoning, so further improvements are possible.

'''